In [ ]:
# Check if interactive widgets are available
try:
    import ipywidgets
    print("✓ ipywidgets available - interactive plots should work")
    
    # Try to enable interactive matplotlib
    %matplotlib widget
    interactive_available = True
except ImportError:
    print("⚠ ipywidgets not available - using static plots")
    print("Install with: pip install ipywidgets")
    %matplotlib inline
    interactive_available = False

# Test basic plotting
import matplotlib.pyplot as plt
import numpy as np

# Simple test plot
fig = plt.figure(figsize=(8, 6))
if interactive_available:
    ax = fig.add_subplot(111, projection='3d')
    
    # Create test data
    t = np.linspace(0, 4*np.pi, 100)
    x = np.cos(t)
    y = np.sin(t) 
    z = t
    
    ax.plot(x, y, z, 'b-')
    ax.set_title('Test Interactive 3D Plot - Try rotating with mouse!')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    
else:
    ax = plt.subplot(111)
    x = np.linspace(0, 10, 100)
    ax.plot(x, np.sin(x))
    ax.set_title('Static 2D Plot')

plt.show()

In [ ]:
import os
import h5py
import mediapy as media
import matplotlib.pyplot as plt
import numpy as np

def load_hdf5(dataset_path):
    if not os.path.isfile(dataset_path):
        print(f'Dataset does not exist at \n{dataset_path}\n')
        exit()

    with h5py.File(dataset_path, 'r') as root:
        qpos = root['/observations/qpos'][()]
        qvel = root['/observations/qvel'][()]
        image_dict = dict()
        for cam_name in root[f'/observations/images/'].keys():
            image_dict[cam_name] = root[f'/observations/images/{cam_name}'][()]
        
        # Load all action types for QP filter analysis
        action = root['/action'][()]  # Final QP-filtered actions
        action_data = {'action': action}  # Store in dict for easier access
        
        # Load additional action types if available
        if '/action_raw' in root:
            action_data['action_raw'] = root['/action_raw'][()]  # Raw ACT policy (no dataset stats)
        if '/action_processed' in root:
            action_data['action_processed'] = root['/action_processed'][()]  # Post-processed (with dataset stats)
        if '/action_filtered' in root:
            action_data['action_filtered'] = root['/action_filtered'][()]  # QP-filtered (deprecated)
            
        # Load metadata attributes
        attrs = {}
        for key in root.attrs.keys():
            attrs[key] = root.attrs[key]

    return qpos, qvel, action_data, image_dict, attrs


def process_depth_for_visualization(depth_mm, manipulation_focus=True):
    """
    Convert depth images from millimeters to visualization-ready format.
    
    Args:
        depth_mm: Depth images in millimeters (uint16)
        manipulation_focus: If True, focus on A4-sized working area (0.3-1.0m range)
    
    Returns:
        depth_vis: Normalized depth for visualization (0-255, closer=brighter)
        depth_stats: Dictionary with depth statistics
    """
    # Convert from mm to meters
    depth_m = depth_mm.astype(np.float32) / 1000.0
    
    if manipulation_focus:
        # Focus on A4 working area: 0.3-1.0m range
        min_depth, max_depth = 0.3, 1.0
        title_suffix = "(manipulation zone)"
    else:
        # Use full range
        min_depth, max_depth = depth_m.min(), depth_m.max()
        title_suffix = "(full range)"
    
    # Clip and normalize depth
    depth_clipped = np.clip(depth_m, min_depth, max_depth)
    depth_normalized = 1.0 - (depth_clipped - min_depth) / (max_depth - min_depth)
    depth_vis = (depth_normalized * 255).astype(np.uint8)
    
    # Calculate statistics
    stats = {
        'min_depth_m': depth_m.min(),
        'max_depth_m': depth_m.max(), 
        'mean_depth_m': depth_m.mean(),
        'manipulation_mean_m': np.mean(depth_m[(depth_m >= 0.3) & (depth_m <= 1.0)]),
        'title_suffix': title_suffix,
        'vis_range': f"{min_depth:.1f}-{max_depth:.1f}m"
    }
    
    return depth_vis, stats

def display_camera_data(image_dict, fps=30):
    """Display both RGB and depth camera data with appropriate processing."""
    
    rgb_cameras = []
    depth_cameras = []
    
    # Separate RGB and depth cameras
    for cam_name, image_data in image_dict.items():
        if cam_name.endswith('_depth'):
            depth_cameras.append((cam_name, image_data))
        else:
            rgb_cameras.append((cam_name, image_data))
    
    print(f"📷 Found {len(rgb_cameras)} RGB cameras and {len(depth_cameras)} depth cameras")
    
    # Display RGB cameras
    for cam_name, image_list in rgb_cameras:
        print(f"\n🎬 RGB Camera: {cam_name}")
        print(f"   Shape: {image_list.shape}, dtype: {image_list.dtype}")
        media.show_video(image_list, fps=fps)
    
    # Display depth cameras  
    for cam_name, depth_list in depth_cameras:
        print(f"\n📏 Depth Camera: {cam_name}")
        print(f"   Shape: {depth_list.shape}, dtype: {depth_list.dtype}")
        
        # Process first frame for statistics
        depth_vis, stats = process_depth_for_visualization(depth_list, manipulation_focus=True)
        
        print(f"   Depth range: {stats['min_depth_m']:.3f} - {stats['max_depth_m']:.3f}m")
        print(f"   Mean depth: {stats['mean_depth_m']:.3f}m")
        print(f"   Manipulation area mean: {stats['manipulation_mean_m']:.3f}m")
        print(f"   Visualization range: {stats['vis_range']}")
        
        # Show depth video with processed visualization
        media.show_video(depth_vis, fps=fps)

In [ ]:
# Load and display episode with RGB + Depth cameras
data_file = r'/home/hk/Documents/ACT_Shaka/act-main/act/test_pointcloud/episode_0.hdf5'
# data_file = 'data/demo/trained.hdf5'

qpos, qvel, action, image_dict = load_hdf5(dataset_path=data_file)

print(f"📊 Episode Statistics:")
print(f"   Duration: {len(action)} timesteps ({len(action)*0.02:.1f}s at 50Hz)")
print(f"   Action space: {action.shape}")
print(f"   Joint positions: {qpos.shape}")
print(f"   Available cameras: {list(image_dict.keys())}")

# Display all camera feeds with appropriate processing
display_camera_data(image_dict, fps=30)

In [ ]:
# Load and display episode with RGB + Depth cameras
data_file = r'/home/hk/Documents/ACT_Shaka/act-main/act/dataset_dir/sim_episodes/episode_3.hdf5'
# data_file = 'data/demo/trained.hdf5'

qpos, qvel, action, image_dict = load_hdf5(dataset_path=data_file)

print(f"📊 Episode Statistics:")
print(f"   Duration: {len(action)} timesteps ({len(action)*0.02:.1f}s at 50Hz)")
print(f"   Action space: {action.shape}")
print(f"   Joint positions: {qpos.shape}")
print(f"   Available cameras: {list(image_dict.keys())}")

# Display all camera feeds with appropriate processing
display_camera_data(image_dict, fps=30)

In [ ]:
# Alternative: Static plots with multiple viewing angles
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
from pointcloud_helpers import load_pointcloud_from_hdf5

# Load the same data
points_camera, points_world, num_points = load_pointcloud_from_hdf5('/home/hk/Documents/ACT_Shaka/act-main/act/dataset_dir/sim_episodes/episode_3.hdf5')

# Create multiple static views
fig = plt.figure(figsize=(15, 10))

# Subsample for faster plotting
max_points = 5000
if len(points_camera) > max_points:
    indices = np.random.choice(len(points_camera), max_points, replace=False)
    points = points_camera[indices]
else:
    points = points_camera

# View 1: Default perspective
ax1 = fig.add_subplot(2, 3, 1, projection='3d')
scatter = ax1.scatter(points[:, 0], points[:, 1], points[:, 2], c=points[:, 2], s=1, cmap='viridis')
ax1.set_title('Default View')
ax1.set_xlabel('X (width)')
ax1.set_ylabel('Y (depth)')
ax1.set_zlabel('Z (height)')
ax1.view_init(elev=20, azim=45)

# View 2: Top-down view
ax2 = fig.add_subplot(2, 3, 2, projection='3d')
ax2.scatter(points[:, 0], points[:, 1], points[:, 2], c=points[:, 2], s=1, cmap='viridis')
ax2.set_title('Top-Down View')
ax2.set_xlabel('X (width)')
ax2.set_ylabel('Y (depth)')
ax2.set_zlabel('Z (height)')
ax2.view_init(elev=90, azim=0)

# View 3: Side view
ax3 = fig.add_subplot(2, 3, 3, projection='3d')
ax3.scatter(points[:, 0], points[:, 1], points[:, 2], c=points[:, 2], s=1, cmap='viridis')
ax3.set_title('Side View')
ax3.set_xlabel('X (width)')
ax3.set_ylabel('Y (depth)')
ax3.set_zlabel('Z (height)')
ax3.view_init(elev=0, azim=0)

# View 4: Front view
ax4 = fig.add_subplot(2, 3, 4, projection='3d')
ax4.scatter(points[:, 0], points[:, 1], points[:, 2], c=points[:, 2], s=1, cmap='viridis')
ax4.set_title('Front View')
ax4.set_xlabel('X (width)')
ax4.set_ylabel('Y (depth)')
ax4.set_zlabel('Z (height)')
ax4.view_init(elev=0, azim=90)

# View 5: XY projection (2D)
ax5 = fig.add_subplot(2, 3, 5)
ax5.scatter(points[:, 0], points[:, 1], c=points[:, 2], s=1, cmap='viridis')
ax5.set_title('XY Projection (Top View)')
ax5.set_xlabel('X (width)')
ax5.set_ylabel('Y (depth)')

# View 6: XZ projection (2D)
ax6 = fig.add_subplot(2, 3, 6)
ax6.scatter(points[:, 0], points[:, 2], c=points[:, 1], s=1, cmap='viridis')
ax6.set_title('XZ Projection (Side View)')
ax6.set_xlabel('X (width)')
ax6.set_ylabel('Z (height)')

plt.tight_layout()
plt.show()

print(f"\nPoint cloud contains {num_points} points")
print(f"X range: {points[:, 0].min():.3f} to {points[:, 0].max():.3f} meters")
print(f"Y range: {points[:, 1].min():.3f} to {points[:, 1].max():.3f} meters") 
print(f"Z range: {points[:, 2].min():.3f} to {points[:, 2].max():.3f} meters")

In [ ]:
# Analyze point clouds at 5 strategic timeframes during manipulation
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
from pointcloud_helpers import load_pointcloud_from_hdf5
import h5py

def analyze_strategic_timeframes(dataset_path, max_timestep=250, num_frames=5):
    """
    Analyze point clouds at strategic timeframes during robot manipulation.
    
    Args:
        dataset_path: Path to HDF5 episode file
        max_timestep: Maximum timestep to consider (default 250)
        num_frames: Number of strategic frames to analyze (default 5)
    """
    
    # First, check how many timesteps are available
    with h5py.File(dataset_path, 'r') as f:
        total_timesteps = f['action'].shape[0]
        available_cameras = list(f['observations/pointclouds/'].keys())
    
    print(f"📊 Episode Analysis:")
    print(f"   Total timesteps: {total_timesteps}")
    print(f"   Analyzing first {min(max_timestep, total_timesteps)} timesteps")
    print(f"   Available cameras: {available_cameras}")
    
    # Calculate strategic timeframes
    end_step = min(max_timestep, total_timesteps)
    strategic_steps = np.linspace(0, end_step-1, num_frames, dtype=int)
    
    print(f"   Strategic timeframes: {strategic_steps}")
    
    # Load point clouds for each strategic timeframe
    camera_name = available_cameras[0]  # Use first available camera
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    
    point_clouds = []
    
    for i, timestep in enumerate(strategic_steps):
        try:
            points_camera, points_world, num_points = load_pointcloud_from_hdf5(
                dataset_path, camera_name=camera_name, timestep=timestep
            )
            point_clouds.append((timestep, points_camera, num_points))
            
            # Subsample for visualization
            max_points = 3000
            if len(points_camera) > max_points:
                indices = np.random.choice(len(points_camera), max_points, replace=False)
                points = points_camera[indices]
            else:
                points = points_camera
            
            # 3D plot for this timeframe
            ax = fig.add_subplot(2, 3, i+1, projection='3d')
            scatter = ax.scatter(points[:, 0], points[:, 1], points[:, 2], 
                               c=points[:, 2], s=1, cmap='viridis')
            
            ax.set_title(f'Timestep {timestep}\n({num_points} points)')
            ax.set_xlabel('X (width)')
            ax.set_ylabel('Y (depth)')
            ax.set_zlabel('Z (height)')
            ax.view_init(elev=20, azim=45)
            
            # Set consistent axis limits for comparison
            if i == 0:
                xlims = [points[:, 0].min(), points[:, 0].max()]
                ylims = [points[:, 1].min(), points[:, 1].max()]
                zlims = [points[:, 2].min(), points[:, 2].max()]
            
            ax.set_xlim(xlims)
            ax.set_ylim(ylims)
            ax.set_zlim(zlims)
            
        except Exception as e:
            print(f"⚠ Error loading timestep {timestep}: {e}")
    
    # Use the last subplot for statistics
    ax_stats = axes[-1]
    ax_stats.axis('off')
    
    # Plot point count over time
    if point_clouds:
        steps = [pc[0] for pc in point_clouds]
        counts = [pc[2] for pc in point_clouds]
        
        ax_stats.plot(steps, counts, 'bo-', linewidth=2, markersize=8)
        ax_stats.set_xlabel('Timestep')
        ax_stats.set_ylabel('Point Count')
        ax_stats.set_title('Point Cloud Density\nOver Time')
        ax_stats.grid(True)
        
        # Add annotations
        for step, count in zip(steps, counts):
            ax_stats.annotate(f'{count}', (step, count), 
                            textcoords="offset points", xytext=(0,10), ha='center')
    
    plt.tight_layout()
    plt.show()
    
    # Print detailed statistics
    print(f"\n📈 Temporal Analysis Results:")
    print("-" * 60)
    for timestep, points, num_points in point_clouds:
        print(f"Timestep {timestep:3d}: {num_points:5d} points | "
              f"X: [{points[:, 0].min():.3f}, {points[:, 0].max():.3f}] | "
              f"Y: [{points[:, 1].min():.3f}, {points[:, 1].max():.3f}] | "
              f"Z: [{points[:, 2].min():.3f}, {points[:, 2].max():.3f}]")
    
    return point_clouds

# Run the analysis
dataset_path = '/home/hk/Documents/ACT_Shaka/act-main/act/dataset_dir/sim_episodes/episode_3.hdf5'
strategic_analysis = analyze_strategic_timeframes(dataset_path, max_timestep=250, num_frames=5)

In [ ]:
# Create animation-style sequence showing manipulation progression (PROPERLY FIXED ORIENTATION)
def create_manipulation_sequence_fixed(dataset_path, max_timestep=250, step_size=20):
    """
    Create a sequence of point cloud snapshots showing manipulation progression.
    PROPERLY fixed to show correct orientation (table at bottom, objects on top).
    """
    
    with h5py.File(dataset_path, 'r') as f:
        total_timesteps = f['action'].shape[0]
        camera_name = list(f['observations/pointclouds/'].keys())[0]
    
    end_step = min(max_timestep, total_timesteps)
    timesteps = list(range(0, end_step, step_size))
    
    # Calculate grid dimensions
    n_cols = 5
    n_rows = (len(timesteps) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3*n_rows))
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    
    sequence_data = []
    
    for i, timestep in enumerate(timesteps):
        row = i // n_cols
        col = i % n_cols
        
        try:
            points_camera, points_world, num_points = load_pointcloud_from_hdf5(
                dataset_path, camera_name=camera_name, timestep=timestep
            )
            
            sequence_data.append((timestep, points_camera, num_points))
            
            # Subsample for performance
            max_points = 2000
            if len(points_camera) > max_points:
                indices = np.random.choice(len(points_camera), max_points, replace=False)
                points = points_camera[indices]
            else:
                points = points_camera
            
            if row < n_rows and col < n_cols:
                ax = axes[row, col] if n_rows > 1 else axes[col]
                
                # PROPER FIX: Simply invert Y coordinates (no axis inversion)
                scatter = ax.scatter(points[:, 0], -points[:, 1], c=points[:, 2], s=2, cmap='viridis')
                ax.set_title(f'Step {timestep}')
                ax.set_xlabel('X (width)')
                ax.set_ylabel('Y (depth)')  # Normal label since we flipped the data
                ax.set_aspect('equal')
                
                # DON'T invert axis - we already flipped the data
                # ax.invert_yaxis()  # REMOVED - this was causing the double flip
                
                # Add point count annotation
                ax.text(0.02, 0.98, f'{num_points} pts', transform=ax.transAxes, 
                       verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
        
        except Exception as e:
            print(f"⚠ Error at timestep {timestep}: {e}")
    
    # Hide empty subplots
    for i in range(len(timesteps), n_rows * n_cols):
        row = i // n_cols
        col = i % n_cols
        if row < n_rows and col < n_cols:
            ax = axes[row, col] if n_rows > 1 else axes[col]
            ax.axis('off')
    
    plt.suptitle(f'Manipulation Sequence: Steps 0-{end_step} (every {step_size} steps) - PROPERLY FIXED', fontsize=16)
    plt.tight_layout()
    plt.show()
    
    return sequence_data

# Run properly corrected sequence analysis
sequence_data = create_manipulation_sequence_fixed(dataset_path, max_timestep=250, step_size=25)

In [ ]:
# Load and display episode with RGB + Depth cameras
data_file = r'/home/hk/Documents/ACT_Shaka/act-main/act/dataset_dir/real_episodes/episode_3.hdf5'
# data_file = 'data/demo/trained.hdf5'

qpos, qvel, action, image_dict = load_hdf5(dataset_path=data_file)

print(f"📊 Episode Statistics:")
print(f"   Duration: {len(action)} timesteps ({len(action)*0.02:.1f}s at 50Hz)")
print(f"   Action space: {action.shape}")
print(f"   Joint positions: {qpos.shape}")
print(f"   Available cameras: {list(image_dict.keys())}")

# Display all camera feeds with appropriate processing
display_camera_data(image_dict, fps=30)

In [ ]:
# Alternative: Static plots with multiple viewing angles
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
from pointcloud_helpers import load_pointcloud_from_hdf5

# Load the same data
points_camera, points_world, num_points = load_pointcloud_from_hdf5('/home/hk/Documents/ACT_Shaka/act-main/act/dataset_dir/real_episodes/episode_3.hdf5')

# Create multiple static views
fig = plt.figure(figsize=(15, 10))

# Subsample for faster plotting
max_points = 5000
if len(points_camera) > max_points:
    indices = np.random.choice(len(points_camera), max_points, replace=False)
    points = points_camera[indices]
else:
    points = points_camera

# View 1: Default perspective
ax1 = fig.add_subplot(2, 3, 1, projection='3d')
scatter = ax1.scatter(points[:, 0], points[:, 1], points[:, 2], c=points[:, 2], s=1, cmap='viridis')
ax1.set_title('Default View')
ax1.set_xlabel('X (width)')
ax1.set_ylabel('Y (depth)')
ax1.set_zlabel('Z (height)')
ax1.view_init(elev=20, azim=45)

# View 2: Top-down view
ax2 = fig.add_subplot(2, 3, 2, projection='3d')
ax2.scatter(points[:, 0], points[:, 1], points[:, 2], c=points[:, 2], s=1, cmap='viridis')
ax2.set_title('Top-Down View')
ax2.set_xlabel('X (width)')
ax2.set_ylabel('Y (depth)')
ax2.set_zlabel('Z (height)')
ax2.view_init(elev=90, azim=0)

# View 3: Side view
ax3 = fig.add_subplot(2, 3, 3, projection='3d')
ax3.scatter(points[:, 0], points[:, 1], points[:, 2], c=points[:, 2], s=1, cmap='viridis')
ax3.set_title('Side View')
ax3.set_xlabel('X (width)')
ax3.set_ylabel('Y (depth)')
ax3.set_zlabel('Z (height)')
ax3.view_init(elev=0, azim=0)

# View 4: Front view
ax4 = fig.add_subplot(2, 3, 4, projection='3d')
ax4.scatter(points[:, 0], points[:, 1], points[:, 2], c=points[:, 2], s=1, cmap='viridis')
ax4.set_title('Front View')
ax4.set_xlabel('X (width)')
ax4.set_ylabel('Y (depth)')
ax4.set_zlabel('Z (height)')
ax4.view_init(elev=0, azim=90)

# View 5: XY projection (2D)
ax5 = fig.add_subplot(2, 3, 5)
ax5.scatter(points[:, 0], points[:, 1], c=points[:, 2], s=1, cmap='viridis')
ax5.set_title('XY Projection (Top View)')
ax5.set_xlabel('X (width)')
ax5.set_ylabel('Y (depth)')

# View 6: XZ projection (2D)
ax6 = fig.add_subplot(2, 3, 6)
ax6.scatter(points[:, 0], points[:, 2], c=points[:, 1], s=1, cmap='viridis')
ax6.set_title('XZ Projection (Side View)')
ax6.set_xlabel('X (width)')
ax6.set_ylabel('Z (height)')

plt.tight_layout()
plt.show()

print(f"\nPoint cloud contains {num_points} points")
print(f"X range: {points[:, 0].min():.3f} to {points[:, 0].max():.3f} meters")
print(f"Y range: {points[:, 1].min():.3f} to {points[:, 1].max():.3f} meters") 
print(f"Z range: {points[:, 2].min():.3f} to {points[:, 2].max():.3f} meters")

In [ ]:
# Analyze point clouds at 5 strategic timeframes during manipulation
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
from pointcloud_helpers import load_pointcloud_from_hdf5
import h5py

def analyze_strategic_timeframes(dataset_path, max_timestep=250, num_frames=5):
    """
    Analyze point clouds at strategic timeframes during robot manipulation.
    
    Args:
        dataset_path: Path to HDF5 episode file
        max_timestep: Maximum timestep to consider (default 250)
        num_frames: Number of strategic frames to analyze (default 5)
    """
    
    # First, check how many timesteps are available
    with h5py.File(dataset_path, 'r') as f:
        total_timesteps = f['action'].shape[0]
        available_cameras = list(f['observations/pointclouds/'].keys())
    
    print(f"📊 Episode Analysis:")
    print(f"   Total timesteps: {total_timesteps}")
    print(f"   Analyzing first {min(max_timestep, total_timesteps)} timesteps")
    print(f"   Available cameras: {available_cameras}")
    
    # Calculate strategic timeframes
    end_step = min(max_timestep, total_timesteps)
    strategic_steps = np.linspace(0, end_step-1, num_frames, dtype=int)
    
    print(f"   Strategic timeframes: {strategic_steps}")
    
    # Load point clouds for each strategic timeframe
    camera_name = available_cameras[0]  # Use first available camera
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    
    point_clouds = []
    
    for i, timestep in enumerate(strategic_steps):
        try:
            points_camera, points_world, num_points = load_pointcloud_from_hdf5(
                dataset_path, camera_name=camera_name, timestep=timestep
            )
            point_clouds.append((timestep, points_camera, num_points))
            
            # Subsample for visualization
            max_points = 3000
            if len(points_camera) > max_points:
                indices = np.random.choice(len(points_camera), max_points, replace=False)
                points = points_camera[indices]
            else:
                points = points_camera
            
            # 3D plot for this timeframe
            ax = fig.add_subplot(2, 3, i+1, projection='3d')
            scatter = ax.scatter(points[:, 0], points[:, 1], points[:, 2], 
                               c=points[:, 2], s=1, cmap='viridis')
            
            ax.set_title(f'Timestep {timestep}\n({num_points} points)')
            ax.set_xlabel('X (width)')
            ax.set_ylabel('Y (depth)')
            ax.set_zlabel('Z (height)')
            ax.view_init(elev=20, azim=45)
            
            # Set consistent axis limits for comparison
            if i == 0:
                xlims = [points[:, 0].min(), points[:, 0].max()]
                ylims = [points[:, 1].min(), points[:, 1].max()]
                zlims = [points[:, 2].min(), points[:, 2].max()]
            
            ax.set_xlim(xlims)
            ax.set_ylim(ylims)
            ax.set_zlim(zlims)
            
        except Exception as e:
            print(f"⚠ Error loading timestep {timestep}: {e}")
    
    # Use the last subplot for statistics
    ax_stats = axes[-1]
    ax_stats.axis('off')
    
    # Plot point count over time
    if point_clouds:
        steps = [pc[0] for pc in point_clouds]
        counts = [pc[2] for pc in point_clouds]
        
        ax_stats.plot(steps, counts, 'bo-', linewidth=2, markersize=8)
        ax_stats.set_xlabel('Timestep')
        ax_stats.set_ylabel('Point Count')
        ax_stats.set_title('Point Cloud Density\nOver Time')
        ax_stats.grid(True)
        
        # Add annotations
        for step, count in zip(steps, counts):
            ax_stats.annotate(f'{count}', (step, count), 
                            textcoords="offset points", xytext=(0,10), ha='center')
    
    plt.tight_layout()
    plt.show()
    
    # Print detailed statistics
    print(f"\n📈 Temporal Analysis Results:")
    print("-" * 60)
    for timestep, points, num_points in point_clouds:
        print(f"Timestep {timestep:3d}: {num_points:5d} points | "
              f"X: [{points[:, 0].min():.3f}, {points[:, 0].max():.3f}] | "
              f"Y: [{points[:, 1].min():.3f}, {points[:, 1].max():.3f}] | "
              f"Z: [{points[:, 2].min():.3f}, {points[:, 2].max():.3f}]")
    
    return point_clouds

# Run the analysis
dataset_path = '/home/hk/Documents/ACT_Shaka/act-main/act/dataset_dir/real_episodes/episode_3.hdf5'
strategic_analysis = analyze_strategic_timeframes(dataset_path, max_timestep=250, num_frames=5)

In [ ]:
# Create animation-style sequence showing manipulation progression (PROPERLY FIXED ORIENTATION)
def create_manipulation_sequence_fixed(dataset_path, max_timestep=250, step_size=20):
    """
    Create a sequence of point cloud snapshots showing manipulation progression.
    PROPERLY fixed to show correct orientation (table at bottom, objects on top).
    """
    
    with h5py.File(dataset_path, 'r') as f:
        total_timesteps = f['action'].shape[0]
        camera_name = list(f['observations/pointclouds/'].keys())[0]
    
    end_step = min(max_timestep, total_timesteps)
    timesteps = list(range(0, end_step, step_size))
    
    # Calculate grid dimensions
    n_cols = 5
    n_rows = (len(timesteps) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3*n_rows))
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    
    sequence_data = []
    
    for i, timestep in enumerate(timesteps):
        row = i // n_cols
        col = i % n_cols
        
        try:
            points_camera, points_world, num_points = load_pointcloud_from_hdf5(
                dataset_path, camera_name=camera_name, timestep=timestep
            )
            
            sequence_data.append((timestep, points_camera, num_points))
            
            # Subsample for performance
            max_points = 2000
            if len(points_camera) > max_points:
                indices = np.random.choice(len(points_camera), max_points, replace=False)
                points = points_camera[indices]
            else:
                points = points_camera
            
            if row < n_rows and col < n_cols:
                ax = axes[row, col] if n_rows > 1 else axes[col]
                
                # PROPER FIX: Simply invert Y coordinates (no axis inversion)
                scatter = ax.scatter(points[:, 0], -points[:, 1], c=points[:, 2], s=2, cmap='viridis')
                ax.set_title(f'Step {timestep}')
                ax.set_xlabel('X (width)')
                ax.set_ylabel('Y (depth)')  # Normal label since we flipped the data
                ax.set_aspect('equal')
                
                # DON'T invert axis - we already flipped the data
                # ax.invert_yaxis()  # REMOVED - this was causing the double flip
                
                # Add point count annotation
                ax.text(0.02, 0.98, f'{num_points} pts', transform=ax.transAxes, 
                       verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
        
        except Exception as e:
            print(f"⚠ Error at timestep {timestep}: {e}")
    
    # Hide empty subplots
    for i in range(len(timesteps), n_rows * n_cols):
        row = i // n_cols
        col = i % n_cols
        if row < n_rows and col < n_cols:
            ax = axes[row, col] if n_rows > 1 else axes[col]
            ax.axis('off')
    
    plt.suptitle(f'Manipulation Sequence: Steps 0-{end_step} (every {step_size} steps) - PROPERLY FIXED', fontsize=16)
    plt.tight_layout()
    plt.show()
    
    return sequence_data

# Run properly corrected sequence analysis
sequence_data = create_manipulation_sequence_fixed(dataset_path, max_timestep=250, step_size=25)

In [ ]:
# Load and analyze episode with RGB + Depth cameras and all action types
data_file = r'/home/hk/Documents/ACT_Shaka/act-main/act/ckpt_sim_pick_cube_teleop/eval_data/episode_24.hdf5'
# data_file = 'data/demo/trained.hdf5'

qpos, qvel, action_data, image_dict, attrs = load_hdf5(dataset_path=data_file)

print(f"📊 Episode Statistics:")
print(f"   Duration: {len(action_data['action'])} timesteps ({len(action_data['action'])*0.02:.1f}s at 50Hz)")
print(f"   Action space: {action_data['action'].shape}")
print(f"   Joint positions: {qpos.shape}")
print(f"   Available cameras: {list(image_dict.keys())}")
print(f"\n📋 Episode Metadata:")
for key, value in attrs.items():
    if key == 'obstacle_height' and value is not None:
        print(f"   {key}: {value:.3f}m")
    else:
        print(f"   {key}: {value}")

print(f"\n🎯 Action Types Available:")
for action_type, data in action_data.items():
    print(f"   {action_type}: {data.shape}")
    # Display all camera feeds with appropriate processing
display_camera_data(image_dict, fps=30)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_action_comparison(action_data, attrs, joint_names=None):
    """
    Plot comparison of different action types to analyze QP filter effects.
    """
    if joint_names is None:
        joint_names = [f'Joint {i+1}' for i in range(action_data['action'].shape[1])]
    
    # Focus on key action types only
    key_action_types = ['action_processed', 'action', 'action_filtered']
    available_types = [t for t in key_action_types if t in action_data]
    print(f"Plotting action comparison for: {available_types}")
    
    # Create figure with subplots for each joint
    n_joints = action_data['action'].shape[1]
    n_cols = min(3, n_joints)
    n_rows = (n_joints + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
    if n_joints == 1:
        axes = [axes]
    elif n_rows == 1:
        axes = axes.reshape(1, -1)
    
    # Color scheme for key action types
    colors = {
        'action_processed': "#CD4E69",  # Teal - Post-processed
        'action': "#8245D100",  # Blue - Final filtered actions
        'action_filtered': "#1D0BBE9B"  # Green - QP-filtered (if separate)
    }
    
    labels = {
        'action_processed': 'Post-processed (with dataset stats)', 
        'action': 'Final filtered actions',
        'action_filtered': 'QP-filtered'
    }
    
    timesteps = np.arange(len(action_data['action']))
    
    for joint_idx in range(n_joints):
        row = joint_idx // n_cols
        col = joint_idx % n_cols
        ax = axes[row, col] if n_rows > 1 else axes[col]
        
        # Plot each available action type
        for action_type in available_types:
            if action_type in action_data:
                ax.plot(timesteps, action_data[action_type][:, joint_idx], 
                       color=colors.get(action_type, 'gray'), 
                       label=labels.get(action_type, action_type),
                       linewidth=1.5, alpha=0.8)
        
        ax.set_title(f'{joint_names[joint_idx]} (Joint {joint_idx+1})')
        ax.set_xlabel('Timestep')
        ax.set_ylabel('Joint Position (rad)')
        ax.grid(True, alpha=0.3)
        ax.legend()
    
    # Hide unused subplots
    for joint_idx in range(n_joints, n_rows * n_cols):
        row = joint_idx // n_cols
        col = joint_idx % n_cols
        ax = axes[row, col] if n_rows > 1 else axes[col]
        ax.set_visible(False)
    
    # Add title with metadata
    obstacle_height = attrs.get('obstacle_height', 'N/A')
    qp_enabled = attrs.get('qp_filter_enabled', False)
    title = f"Action Comparison - Obstacle Height: {obstacle_height}m, QP Filter: {'Yes' if qp_enabled else 'No'}"
    fig.suptitle(title, fontsize=16, y=0.98)
    
    plt.tight_layout()
    plt.subplots_adjust(top=0.93)
    plt.show()

# Perform focused action analysis
joint_names = ['Shoulder Pan', 'Shoulder Lift', 'Elbow', 'Wrist 1', 'Wrist 2', 'Wrist 3', 'Gripper']
plot_action_comparison(action_data, attrs, joint_names[:action_data['action'].shape[1]])

In [ ]:
# plot action
plt.figure()
plt.plot(action, label=list(range(1, 8)))
plt.xlabel('step')
plt.ylabel('angle [rad]')
plt.title('joint positions')
plt.legend()

In [ ]:
# plot action
plt.figure()
plt.plot(action_filtered, label=list(range(1, 8)))
plt.xlabel('step')
plt.ylabel('angle [rad]')
plt.title('joint positions')
plt.legend()

In [ ]:
# plot qpos
plt.figure()
plt.plot(qpos, label=list(range(1, 7)))
plt.xlabel('step')
plt.ylabel('angle [rad]')
plt.title('joint positions')
plt.legend()


In [ ]:
# plot action
plt.figure()
plt.plot(action, label=list(range(1, 7)))
plt.xlabel('step')
plt.ylabel('angle [rad]')
plt.title('joint positions')
plt.legend()

In [ ]:
# play cam video
data_file = r'C:\Users\Administrator\Documents\transformer\ACT-Shaka\data\evaluation\task3c\eval_result_task3c_20250914_183648.hdf5'
# data_file = 'data/demo/trained.hdf5'
qpos, qvel, action, image_dict = load_hdf5(dataset_path=data_file)
for cam_name, image_list in image_dict.items():
    media.show_video(image_list, fps=30)

In [ ]:
# plot action
plt.figure()
plt.plot(action, label=list(range(1, 7)))
plt.xlabel('step')
plt.ylabel('angle [rad]')
plt.title('joint positions')
plt.legend()


In [ ]:
# play cam video
data_file = r'C:\Users\Administrator\Documents\transformer\ACT-Shaka\data\task4c\episode_5.hdf5'
# data_file = 'data/demo/trained.hdf5'
qpos, qvel, action, image_dict = load_hdf5(dataset_path=data_file)

print(f"Dataset contains cameras: {list(image_dict.keys())}")

for cam_name, image_list in image_dict.items():
    print(f"\n📹 Camera: {cam_name}")
    print(f"   Shape: {image_list.shape}")
    print(f"   Frames: {len(image_list)}")
    
    # Show video with camera name in title
    media.show_video(image_list, fps=30, title=f"Camera: {cam_name}")

In [ ]:
# plot action
plt.figure()
plt.plot(action, label=list(range(1, 7)))
plt.xlabel('step')
plt.ylabel('angle [rad]')
plt.title('joint positions')
plt.legend()


In [ ]:
# play cam video
data_file = r'C:\Users\Administrator\Documents\transformer\ACT-Shaka\data\evaluation\task4c\eval_result_task4c_20250915_203958.hdf5'
# data_file = 'data/demo/trained.hdf5'
qpos, qvel, action, image_dict = load_hdf5(dataset_path=data_file)
for cam_name, image_list in image_dict.items():
    media.show_video(image_list, fps=30)

In [ ]:
# plot qpos
plt.figure()
plt.plot(action, label=list(range(1, 7)))
plt.xlabel('step')
plt.ylabel('angle [rad]')
plt.title('joint positions')
plt.legend()


In [ ]:
# play cam video
data_file = r'C:\Users\Administrator\Documents\transformer\ACT-Shaka\data\evaluation\task5C\eval_result_task5c_20250916_234909.hdf5'
# data_file = 'data/demo/trained.hdf5'
qpos, qvel, action, image_dict = load_hdf5(dataset_path=data_file)
for cam_name, image_list in image_dict.items():
    media.show_video(image_list, fps=30)

In [ ]:
# plot qpos
plt.figure()
plt.plot(action, label=list(range(1, 7)))
plt.xlabel('step')
plt.ylabel('angle [rad]')
plt.title('joint positions')
plt.legend()


In [ ]:
# play cam video
data_file = r'C:\Users\Administrator\Documents\transformer\ACT-Shaka\data\task6c\episode_45.hdf5'
# data_file = 'data/demo/trained.hdf5'
qpos, qvel, action, image_dict = load_hdf5(dataset_path=data_file)
for cam_name, image_list in image_dict.items():
    media.show_video(image_list, fps=30)

In [ ]:
# plot qpos
plt.figure()
plt.plot(qpos, label=list(range(1, 7)))
plt.xlabel('step')
plt.ylabel('angle [rad]')
plt.title('joint positions')
plt.legend()


In [ ]:
# plot qpos
plt.figure()
plt.plot(action, label=list(range(1, 7)))
plt.xlabel('step')
plt.ylabel('angle [rad]')
plt.title('joint positions')
plt.legend()


In [ ]:
# play cam video
data_file = r'C:\Users\Administrator\Documents\transformer\ACT-Shaka\data\evaluation\task6C\eval_result_task6c_20251008_230631.hdf5'
# data_file = 'data/demo/trained.hdf5'
qpos, qvel, action, image_dict = load_hdf5(dataset_path=data_file)
for cam_name, image_list in image_dict.items():
    media.show_video(image_list, fps=30)

In [ ]:
# plot qpos
plt.figure()
plt.plot(qpos, label=list(range(1, 7)))
plt.xlabel('step')
plt.ylabel('angle [rad]')
plt.title('joint positions')
plt.legend()


In [ ]:
# play cam video
data_file = r'C:\Users\Administrator\Documents\transformer\ACT-Shaka\data\evaluation\task6c\eval_result_task6c_20251012_161642.hdf5'
# data_file = 'data/demo/trained.hdf5'
qpos, qvel, action, image_dict = load_hdf5(dataset_path=data_file)
for cam_name, image_list in image_dict.items():
    media.show_video(image_list, fps=30)

In [ ]:
# plot qpos
plt.figure()
plt.plot(action, label=list(range(1, 7)))
plt.xlabel('step')
plt.ylabel('angle [rad]')
plt.title('joint positions')
plt.legend()


In [ ]:
# play cam video
data_file = r'C:\Users\Administrator\Documents\transformer\ACT-Shaka\data\evaluation\task6c\eval_result_task6c_20251012_163955.hdf5'
# data_file = 'data/demo/trained.hdf5'
qpos, qvel, action, image_dict = load_hdf5(dataset_path=data_file)
for cam_name, image_list in image_dict.items():
    media.show_video(image_list, fps=30)

In [ ]:
# plot qpos
plt.figure()
plt.plot(action, label=list(range(1, 7)))
plt.xlabel('step')
plt.ylabel('angle [rad]')
plt.title('joint positions')
plt.legend()


In [ ]:
# play cam video
data_file = r'C:\Users\Administrator\Documents\transformer\ACT-Shaka\data\task6c\episode_58.hdf5'
# data_file = 'data/demo/trained.hdf5'
qpos, qvel, action, image_dict = load_hdf5(dataset_path=data_file)
for cam_name, image_list in image_dict.items():
    media.show_video(image_list, fps=30)

In [ ]:
# plot qpos
plt.figure()
plt.plot(qpos, label=list(range(1, 7)))
plt.xlabel('step')
plt.ylabel('angle [rad]')
plt.title('joint positions')
plt.legend()


In [ ]:
# play cam video
data_file = r'C:\Users\Administrator\Documents\transformer\ACT-Shaka\data\evaluation\task6c\eval_result_task6c_20251013_234103.hdf5'
# data_file = 'data/demo/trained.hdf5'
qpos, qvel, action, image_dict = load_hdf5(dataset_path=data_file)
for cam_name, image_list in image_dict.items():
    media.show_video(image_list, fps=30)

In [ ]:
# plot qpos
plt.figure()
plt.plot(qpos, label=list(range(1, 7)))
plt.xlabel('step')
plt.ylabel('angle [rad]')
plt.title('joint positions')
plt.legend()


In [ ]:
# play cam video
data_file = r'C:\Users\Administrator\Documents\transformer\ACT-Shaka\data\evaluation\task6c\eval_result_task6c_20251014_000005.hdf5'
# data_file = 'data/demo/trained.hdf5'
qpos, qvel, action, image_dict = load_hdf5(dataset_path=data_file)
for cam_name, image_list in image_dict.items():
    media.show_video(image_list, fps=30)

In [ ]:
# plot qpos
plt.figure()
plt.plot(qpos, label=list(range(1, 7)))
plt.xlabel('step')
plt.ylabel('angle [rad]')
plt.title('joint positions')
plt.legend()


In [ ]:
# play cam video
data_file = r'C:\Users\Administrator\Documents\transformer\ACT-Shaka\data\task6c\episode_129.hdf5'
# data_file = 'data/demo/trained.hdf5'
qpos, qvel, action, image_dict = load_hdf5(dataset_path=data_file)
for cam_name, image_list in image_dict.items():
    media.show_video(image_list, fps=30)

In [ ]:
# plot qpos
plt.figure()
plt.plot(qpos, label=list(range(1, 7)))
plt.xlabel('step')
plt.ylabel('angle [rad]')
plt.title('joint positions')
plt.legend()


In [ ]:
# plot qpos
plt.figure()
plt.plot(action, label=list(range(1, 7)))
plt.xlabel('step')
plt.ylabel('angle [rad]')
plt.title('joint positions')
plt.legend()


In [ ]:
# play cam video
data_file = r'C:\Users\Administrator\Documents\transformer\ACT-Shaka\data\task7c\episode_65.hdf5'
# data_file = 'data/demo/trained.hdf5'
qpos, qvel, action, image_dict = load_hdf5(dataset_path=data_file)
for cam_name, image_list in image_dict.items():
    media.show_video(image_list, fps=30)

In [ ]:
# plot qpos
plt.figure()
plt.plot(qpos, label=list(range(1, 7)))
plt.xlabel('step')
plt.ylabel('angle [rad]')
plt.title('joint positions')
plt.legend()


In [ ]:
# plot qpos
plt.figure()
plt.plot(action, label=list(range(1, 7)))
plt.xlabel('step')
plt.ylabel('angle [rad]')
plt.title('joint positions')
plt.legend()


In [ ]:
# play cam video
data_file = r'C:\Users\Administrator\Documents\transformer\ACT-Shaka\data\evaluation\task7c\eval_result_task7c_20251019_233217.hdf5'
# data_file = 'data/demo/trained.hdf5'
qpos, qvel, action, image_dict = load_hdf5(dataset_path=data_file)
for cam_name, image_list in image_dict.items():
    media.show_video(image_list, fps=30)

In [ ]:
# play cam video
data_file = r'/home/hk/Documents/ACT_Shaka/act-main/act/single_arm_pick_dataset/episode_60.hdf5'
# data_file = 'data/demo/trained.hdf5'
qpos, qvel, action, image_dict = load_hdf5(dataset_path=data_file)
for cam_name, image_list in image_dict.items():
    media.show_video(image_list, fps=30)

In [ ]:
# plot qpos
plt.figure()
plt.plot(qpos, label=list(range(1, 8)))
plt.xlabel('step')
plt.ylabel('angle [rad]')
plt.title('joint positions')
plt.legend()

In [ ]:
# plot qpos
plt.figure()
plt.plot(action, label=list(range(1, 8)))
plt.xlabel('step')
plt.ylabel('angle [rad]')
plt.title('joint positions')
plt.legend()

In [ ]:
# plot qpos
plt.figure()
plt.plot(action, label=list(range(1, 7)))
plt.xlabel('step')
plt.ylabel('angle [rad]')
plt.title('joint positions')
plt.legend()


In [ ]:
# play cam video
data_file = r'/home/hk/Documents/ACT_Shaka/act-main/act/dataset_dir/real_episodes/episode_1.hdf5'
# data_file = 'data/demo/trained.hdf5'
qpos, qvel, action, image_dict = load_hdf5(dataset_path=data_file)
for cam_name, image_list in image_dict.items():
    media.show_video(image_list, fps=30)

In [ ]:
plt.figure(figsize=(12, 8))
joint_names = ['Joint 1', 'Joint 2', 'Joint 3', 'Joint 4', 'Joint 5', 'Joint 6', 'Gripper']
for i in range(7):  # Changed from 6 to 7
    plt.plot(qpos[:, i], label=joint_names[i])
plt.xlabel('Timestep')
plt.ylabel('Value [rad] / [0-1]')
plt.title('Joint Positions (7-DOF: 6 joints + gripper)')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(12, 8))
joint_names = ['Joint 1', 'Joint 2', 'Joint 3', 'Joint 4', 'Joint 5', 'Joint 6', 'Gripper']
for i in range(7):  # Changed from 6 to 7
    plt.plot(action[:, i], label=joint_names[i])
plt.xlabel('Timestep')
plt.ylabel('Value [rad] / [0-1]')
plt.title('Joint Positions (7-DOF: 6 joints + gripper)')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# play cam video
data_file = r'/home/hk/Documents/ACT_Shaka/act-main/act/dataset_dir/sim_episodes/episode_1.hdf5'
# data_file = 'data/demo/trained.hdf5'
qpos, qvel, action, image_dict = load_hdf5(dataset_path=data_file)
for cam_name, image_list in image_dict.items():
    media.show_video(image_list, fps=30)

In [ ]:
plt.figure(figsize=(12, 8))
joint_names = ['Joint 1', 'Joint 2', 'Joint 3', 'Joint 4', 'Joint 5', 'Joint 6', 'Gripper']
for i in range(7):  # Changed from 6 to 7
    plt.plot(qpos[:, i], label=joint_names[i])
plt.xlabel('Timestep')
plt.ylabel('Value [rad] / [0-1]')
plt.title('Sim Joint Positions (7-DOF: 6 joints + gripper)')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(12, 8))
joint_names = ['Joint 1', 'Joint 2', 'Joint 3', 'Joint 4', 'Joint 5', 'Joint 6', 'Gripper']
for i in range(7):  # Changed from 6 to 7
    plt.plot(action[:, i], label=joint_names[i])
plt.xlabel('Timestep')
plt.ylabel('Value [rad] / [0-1]')
plt.title('Joint Positions (7-DOF: 6 joints + gripper)')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# play cam video
data_file = r'/home/hk/Documents/ACT_Shaka/act-main/act/teleop_data/episode_16.hdf5'
# data_file = 'data/demo/trained.hdf5'
qpos, qvel, action, image_dict = load_hdf5(dataset_path=data_file)
for cam_name, image_list in image_dict.items():
    media.show_video(image_list, fps=30)

In [ ]:
plt.figure(figsize=(12, 8))
joint_names = ['Joint 1', 'Joint 2', 'Joint 3', 'Joint 4', 'Joint 5', 'Joint 6', 'Gripper']
for i in range(7):  # Changed from 6 to 7
    plt.plot(qpos[:, i], label=joint_names[i])
plt.xlabel('Timestep')
plt.ylabel('Value [rad] / [0-1]')
plt.title('Joint Positions (7-DOF: 6 joints + gripper)')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(12, 8))
joint_names = ['Joint 1', 'Joint 2', 'Joint 3', 'Joint 4', 'Joint 5', 'Joint 6', 'Gripper']
for i in range(7):  # Changed from 6 to 7
    plt.plot(action[:, i], label=joint_names[i])
plt.xlabel('Timestep')
plt.ylabel('Value [rad] / [0-1]')
plt.title('Joint Positions (7-DOF: 6 joints + gripper)')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# play cam video
data_file = r'/home/hk/Documents/ACT_Shaka/act-main/act/ckpt_sim_pick_cube_teleop/eval_data/eval_rollout_1_20260213_121212.hdf5'
# data_file = 'data/demo/trained.hdf5'
qpos, qvel, action, image_dict = load_hdf5(dataset_path=data_file)
for cam_name, image_list in image_dict.items():
    media.show_video(image_list, fps=30)

In [ ]:
plt.figure(figsize=(12, 8))
joint_names = ['Joint 1', 'Joint 2', 'Joint 3', 'Joint 4', 'Joint 5', 'Joint 6', 'Gripper']
for i in range(7):  # Changed from 6 to 7
    plt.plot(qpos[:, i], label=joint_names[i])
plt.xlabel('Timestep')
plt.ylabel('Value [rad] / [0-1]')
plt.title('Joint Positions (7-DOF: 6 joints + gripper)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
joint_names = ['Joint 1', 'Joint 2', 'Joint 3', 'Joint 4', 'Joint 5', 'Joint 6', 'Gripper']
for i in range(7):  # Changed from 6 to 7
    plt.plot(action[:, i], label=joint_names[i])
plt.xlabel('Timestep')
plt.ylabel('Value [rad] / [0-1]')
plt.title('Joint Positions (7-DOF: 6 joints + gripper)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# play cam video
data_file = r'/home/hk/Documents/ACT_Shaka/act-main/act/teleop_data/episode_12.hdf5'
# data_file = 'data/demo/trained.hdf5'
qpos, qvel, action, image_dict = load_hdf5(dataset_path=data_file)
for cam_name, image_list in image_dict.items():
    media.show_video(image_list, fps=30)

In [ ]:
plt.figure(figsize=(12, 8))
joint_names = ['Joint 1', 'Joint 2', 'Joint 3', 'Joint 4', 'Joint 5', 'Joint 6', 'Gripper']
for i in range(7):  # Changed from 6 to 7
    plt.plot(action[:, i], label=joint_names[i])
plt.xlabel('Timestep')
plt.ylabel('Value [rad] / [0-1]')
plt.title('Joint Positions (7-DOF: 6 joints + gripper)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
joint_names = ['Joint 1', 'Joint 2', 'Joint 3', 'Joint 4', 'Joint 5', 'Joint 6', 'Gripper']
for i in range(7):  # Changed from 6 to 7
    plt.plot(qpos[:, i], label=joint_names[i])
plt.xlabel('Timestep')
plt.ylabel('Value [rad] / [0-1]')
plt.title('Joint Positions (7-DOF: 6 joints + gripper)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# play cam video
data_file = r'/home/hk/Documents/ACT_Shaka/act-main/act/teleop_data/episode_25.hdf5'
# data_file = 'data/demo/trained.hdf5'
qpos, qvel, action, image_dict = load_hdf5(dataset_path=data_file)
for cam_name, image_list in image_dict.items():
    media.show_video(image_list, fps=30)

In [ ]:
plt.figure(figsize=(12, 8))
joint_names = ['Joint 1', 'Joint 2', 'Joint 3', 'Joint 4', 'Joint 5', 'Joint 6', 'Gripper']
for i in range(7):  # Changed from 6 to 7
    plt.plot(action[:, i], label=joint_names[i])
plt.xlabel('Timestep')
plt.ylabel('Value [rad] / [0-1]')
plt.title('Joint Positions (7-DOF: 6 joints + gripper)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
joint_names = ['Joint 1', 'Joint 2', 'Joint 3', 'Joint 4', 'Joint 5', 'Joint 6', 'Gripper']
for i in range(7):  # Changed from 6 to 7
    plt.plot(qpos[:, i], label=joint_names[i])
plt.xlabel('Timestep')
plt.ylabel('Value [rad] / [0-1]')
plt.title('Joint Positions (7-DOF: 6 joints + gripper)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# play cam video
data_file = r'/home/hk/Documents/ACT_Shaka/act-main/act/ckpt_sim_pick_cube_teleop/eval_data/eval_rollout_3_20260220_110223.hdf5'
# data_file = 'data/demo/trained.hdf5'
qpos, qvel, action, image_dict = load_hdf5(dataset_path=data_file)
for cam_name, image_list in image_dict.items():
    media.show_video(image_list, fps=30)

In [ ]:
plt.figure(figsize=(12, 8))
joint_names = ['Joint 1', 'Joint 2', 'Joint 3', 'Joint 4', 'Joint 5', 'Joint 6', 'Gripper']
for i in range(7):  # Changed from 6 to 7
    plt.plot(action[:, i], label=joint_names[i])
plt.xlabel('Timestep')
plt.ylabel('Value [rad] / [0-1]')
plt.title('Joint Positions (7-DOF: 6 joints + gripper)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
joint_names = ['Joint 1', 'Joint 2', 'Joint 3', 'Joint 4', 'Joint 5', 'Joint 6', 'Gripper']
for i in range(7):  # Changed from 6 to 7
    plt.plot(qpos[:, i], label=joint_names[i])
plt.xlabel('Timestep')
plt.ylabel('Value [rad] / [0-1]')
plt.title('Joint Positions (7-DOF: 6 joints + gripper)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# play cam video
data_file = r'/home/hk/Documents/ACT_Shaka/data/task1/episode_24.hdf5'
# data_file = 'data/demo/trained.hdf5'
qpos, qvel, action, image_dict = load_hdf5(dataset_path=data_file)
for cam_name, image_list in image_dict.items():
    media.show_video(image_list, fps=30)

In [ ]:
plt.figure(figsize=(12, 8))
joint_names = ['Joint 1', 'Joint 2', 'Joint 3', 'Joint 4', 'Joint 5', 'Joint 6', 'Gripper']
for i in range(7):  # Changed from 6 to 7
    plt.plot(action[:, i], label=joint_names[i])
plt.xlabel('Timestep')
plt.ylabel('Value [rad] / [0-1]')
plt.title('Joint Positions (7-DOF: 6 joints + gripper)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
joint_names = ['Joint 1', 'Joint 2', 'Joint 3', 'Joint 4', 'Joint 5', 'Joint 6', 'Gripper']
for i in range(7):  # Changed from 6 to 7
    plt.plot(qpos[:, i], label=joint_names[i])
plt.xlabel('Timestep')
plt.ylabel('Value [rad] / [0-1]')
plt.title('Joint Positions (7-DOF: 6 joints + gripper)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# play cam video
data_file = r'/home/hk/Documents/ACT_Shaka/data/task1/episode_13.hdf5'
# data_file = 'data/demo/trained.hdf5'
qpos, qvel, action, image_dict = load_hdf5(dataset_path=data_file)
for cam_name, image_list in image_dict.items():
    media.show_video(image_list, fps=30)

In [ ]:
plt.figure(figsize=(12, 8))
joint_names = ['Joint 1', 'Joint 2', 'Joint 3', 'Joint 4', 'Joint 5', 'Joint 6', 'Gripper']
for i in range(7):  # Changed from 6 to 7
    plt.plot(action[:, i], label=joint_names[i])
plt.xlabel('Timestep')
plt.ylabel('Value [rad] / [0-1]')
plt.title('Joint Positions (7-DOF: 6 joints + gripper)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
joint_names = ['Joint 1', 'Joint 2', 'Joint 3', 'Joint 4', 'Joint 5', 'Joint 6', 'Gripper']
for i in range(7):  # Changed from 6 to 7
    plt.plot(qpos[:, i], label=joint_names[i])
plt.xlabel('Timestep')
plt.ylabel('Value [rad] / [0-1]')
plt.title('Joint Positions (7-DOF: 6 joints + gripper)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# play cam video
data_file = r'/home/hk/Documents/ACT_Shaka/act-main/act/teleop_data/episode_22.hdf5'
# data_file = 'data/demo/trained.hdf5'
qpos, qvel, action, image_dict = load_hdf5(dataset_path=data_file)
for cam_name, image_list in image_dict.items():
    media.show_video(image_list, fps=30)


In [ ]:
plt.figure(figsize=(12, 8))
joint_names = ['Joint 1', 'Joint 2', 'Joint 3', 'Joint 4', 'Joint 5', 'Joint 6', 'Gripper']
for i in range(7):  # Changed from 6 to 7
    plt.plot(qpos[:, i], label=joint_names[i])
plt.xlabel('Timestep')
plt.ylabel('Value [rad] / [0-1]')
plt.title('Real Joint Positions (7-DOF: 6 joints + gripper)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# play cam video
data_file = r'/home/hk/Documents/ACT_Shaka/act-main/act/ckpt_sim_pick_cube_teleop/eval_data/eval_rollout_0_20260220_170235.hdf5'
# data_file = 'data/demo/trained.hdf5'
qpos, qvel, action, image_dict = load_hdf5(dataset_path=data_file)
for cam_name, image_list in image_dict.items():
    media.show_video(image_list, fps=30)

In [ ]:
plt.figure(figsize=(12, 8))
joint_names = ['Joint 1', 'Joint 2', 'Joint 3', 'Joint 4', 'Joint 5', 'Joint 6', 'Gripper']
for i in range(7):  # Changed from 6 to 7
    plt.plot(qpos[:, i], label=joint_names[i])
plt.xlabel('Timestep')
plt.ylabel('Value [rad] / [0-1]')
plt.title('Real Joint Positions (7-DOF: 6 joints + gripper)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
joint_names = ['Joint 1', 'Joint 2', 'Joint 3', 'Joint 4', 'Joint 5', 'Joint 6', 'Gripper']
for i in range(7):  # Changed from 6 to 7
    plt.plot(action[:, i], label=joint_names[i])
plt.xlabel('Timestep')
plt.ylabel('Value [rad] / [0-1]')
plt.title('Real Joint Positions (7-DOF: 6 joints + gripper)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# play cam video
data_file = r'/home/hk/Documents/ACT_Shaka/act-main/act/teleop_data/episode_0.hdf5'
# data_file = 'data/demo/trained.hdf5'
qpos, qvel, action, image_dict = load_hdf5(dataset_path=data_file)
for cam_name, image_list in image_dict.items():
    media.show_video(image_list, fps=30)

In [ ]:
plt.figure(figsize=(12, 8))
joint_names = ['Joint 1', 'Joint 2', 'Joint 3', 'Joint 4', 'Joint 5', 'Joint 6', 'Gripper']
for i in range(7):  # Changed from 6 to 7
    plt.plot(qpos[:, i], label=joint_names[i])
plt.xlabel('Timestep')
plt.ylabel('Value [rad] / [0-1]')
plt.title(' Joint Positions (7-DOF: 6 joints + gripper)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
joint_names = ['Joint 1', 'Joint 2', 'Joint 3', 'Joint 4', 'Joint 5', 'Joint 6', 'Gripper']
for i in range(7):  # Changed from 6 to 7
    plt.plot(action[:, i], label=joint_names[i])
plt.xlabel('Timestep')
plt.ylabel('Value [rad] / [0-1]')
plt.title('Real Joint Positions (7-DOF: 6 joints + gripper)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# play cam video
data_file = r'/home/hk/Documents/ACT_Shaka/act-main/act/ckpt_sim_pick_cube_teleop/eval_data/eval_rollout_0_20260220_213041.hdf5'
# data_file = 'data/demo/trained.hdf5'
qpos, qvel, action, image_dict = load_hdf5(dataset_path=data_file)
for cam_name, image_list in image_dict.items():
    media.show_video(image_list, fps=30)

In [ ]:
plt.figure(figsize=(12, 8))
joint_names = ['Joint 1', 'Joint 2', 'Joint 3', 'Joint 4', 'Joint 5', 'Joint 6', 'Gripper']
for i in range(7):  # Changed from 6 to 7
    plt.plot(qpos[:, i], label=joint_names[i])
plt.xlabel('Timestep')
plt.ylabel('Value [rad] / [0-1]')
plt.title(' Joint Positions (7-DOF: 6 joints + gripper)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
joint_names = ['Joint 1', 'Joint 2', 'Joint 3', 'Joint 4', 'Joint 5', 'Joint 6', 'Gripper']
for i in range(7):  # Changed from 6 to 7
    plt.plot(action[:, i], label=joint_names[i])
plt.xlabel('Timestep')
plt.ylabel('Value [rad] / [0-1]')
plt.title('Real Joint Positions (7-DOF: 6 joints + gripper)')
plt.legend()
plt.grid(True)
plt.show()